# Limpieza, deduplicacion y creacion de chunks

Objetivo: convertir transcripciones crudas en fragmentos etiquetables, conservando trazabilidad hacia video, canal, tiempo de inicio y hash de texto.

In [ ]:
!pip3 install -q pandas scikit-learn tqdm

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata

import pandas as pd

ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'datos' / 'raw'
INTERIM_DIR = ROOT / 'datos' / 'interim'
PROCESSED_DIR = ROOT / 'datos' / 'processed'

for path in [INTERIM_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_TRANSCRIPTS = RAW_DIR / 'transcripts_raw.jsonl'
print('Entrada esperada:', RAW_TRANSCRIPTS)

## 1. Taxonomia inicial

El etiquetado se plantea como multi-etiqueta. Un mismo chunk puede ser limpio, o puede activar una o mas categorias de moderacion.

In [ ]:
TAXONOMY = pd.DataFrame([
    {'label': 'limpio', 'descripcion': 'No contiene infraccion de moderacion en el fragmento.'},
    {'label': 'lenguaje_ofensivo', 'descripcion': 'Insultos, groserias o ataques verbales no protegidos por cita o analisis.'},
    {'label': 'odio_discriminacion', 'descripcion': 'Ataque o inferiorizacion por origen, etnia, genero, orientacion, religion, discapacidad u otra clase protegida.'},
    {'label': 'acoso_amenaza', 'descripcion': 'Hostigamiento dirigido, amenaza o deseo de dano contra una persona o grupo.'},
    {'label': 'sexual_explicito', 'descripcion': 'Descripcion sexual explicita, acoso sexual o cosificacion no contextual.'},
    {'label': 'violencia', 'descripcion': 'Incitacion, celebracion o descripcion grafica de violencia.'},
    {'label': 'desinformacion_danina', 'descripcion': 'Afirmacion verificable falsa o no sustentada que puede causar dano publico.'},
    {'label': 'privacidad_doxxing', 'descripcion': 'Datos personales sensibles, direcciones, telefonos o identificadores privados.'},
    {'label': 'spam_estafa', 'descripcion': 'Promocion enganosa, fraude, enlaces sospechosos o captacion maliciosa.'},
    {'label': 'contexto_sensible', 'descripcion': 'Fragmento que requiere revision por ironia, cita, parodia, noticia o debate politico.'},
])

TAXONOMY.to_csv(PROCESSED_DIR / 'taxonomia_moderacion.csv', index=False)
TAXONOMY

In [ ]:
def load_jsonl(path):
    if not path.exists():
        print('No existe el archivo:', path)
        return []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def normalize_text(text):
    text = unicodedata.normalize('NFKC', text or '')
    text = text.replace('\n', ' ')
    text = re.sub(r'\[(musica|aplausos|risas|music|applause|laughter)\]', ' ', text, flags=re.I)
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def text_hash(text):
    return hashlib.md5(normalize_text(text).lower().encode('utf-8')).hexdigest()

In [ ]:
def build_chunks(record, target_seconds=45, max_chars=900, min_chars=120):
    chunks = []
    current = []
    start = None
    end = None
    char_count = 0

    for seg in record.get('segments', []):
        seg_text = normalize_text(seg.get('text', ''))
        if not seg_text:
            continue
        seg_start = float(seg.get('start', 0.0))
        seg_end = seg_start + float(seg.get('duration', 0.0))
        if start is None:
            start = seg_start
        end = seg_end
        current.append(seg_text)
        char_count += len(seg_text)

        if (end - start >= target_seconds) or (char_count >= max_chars):
            text = normalize_text(' '.join(current))
            if len(text) >= min_chars:
                chunks.append(make_chunk(record, start, end, text, len(chunks)))
            current, start, end, char_count = [], None, None, 0

    text = normalize_text(' '.join(current))
    if len(text) >= min_chars:
        chunks.append(make_chunk(record, start, end, text, len(chunks)))
    return chunks


def make_chunk(record, start, end, text, idx):
    video_id = record.get('video_id', 'sin_video')
    chunk_id = f'{video_id}_{idx:04d}'
    return {
        'chunk_id': chunk_id,
        'video_id': video_id,
        'channel_id': record.get('channel_id'),
        'channel_title': record.get('channel_title'),
        'video_title': record.get('title'),
        'published_at': record.get('published_at'),
        'start_seconds': round(float(start or 0.0), 2),
        'end_seconds': round(float(end or 0.0), 2),
        'text': text,
        'text_hash': text_hash(text),
        'labels': [],
        'needs_review': True,
        'annotator': '',
        'notes': '',
    }

In [ ]:
records = load_jsonl(RAW_TRANSCRIPTS)
chunks = []
for record in records:
    chunks.extend(build_chunks(record))

chunks_df = pd.DataFrame(chunks)
if not chunks_df.empty:
    chunks_df = chunks_df.drop_duplicates('text_hash').reset_index(drop=True)
    chunks_df.to_csv(PROCESSED_DIR / 'chunks_para_etiquetar.csv', index=False)
    write_jsonl(chunks_df.to_dict(orient='records'), PROCESSED_DIR / 'chunks_para_etiquetar.jsonl')

print('Registros fuente:', len(records))
print('Chunks unicos:', len(chunks_df))
chunks_df.head()

## 2. Checklist de calidad

- Cada chunk debe conservar `video_id`, `channel_title`, `start_seconds` y `end_seconds`.
- Los duplicados se eliminan por `text_hash` despues de normalizar texto.
- Los chunks sin contexto suficiente deben marcarse como `contexto_sensible` o descartarse antes del entrenamiento.
- La categoria `limpio` no debe combinarse con etiquetas de infraccion.